# Lecture 2 (Part 1): Data Structures and NumPy


---

Data types sound like the driest topic in programming. They are also where the most **expensive** mistakes happen: a float compared with `==`, a list multiplied where an array was intended, a slice that silently modified the original data. This notebook is a sequence of small predictions — most of which most people get wrong the first time.


## 2.1 Numbers

Python distinguishes integers (`int`) and real numbers (`float`). The built-in function `type()` tells you what an object is:

In [ ]:
maturity = 10       # years
type(maturity)

In [ ]:
maturity = 10.0     # years
type(maturity)      # now recognized as float

In [ ]:
type(5/4)           # division always produces a float -- even 4/4

### Predict

You charge a client 10 cents, then 20 cents. Does Python agree the total is 30 cents? What does the next cell print — `True` or `False`?

In [ ]:
0.1 + 0.2 == 0.3

In [ ]:
print(0.1 + 0.2)    # look at the actual value

<details>
<summary> <b>Explanation</b></summary>

`False`. Computers store floats in binary, and 0.1 is as impossible to write exactly in binary as 1/3 is in decimal — so tiny rounding errors are baked in (0.1 + 0.2 is stored as 0.30000000000000004). Two important consequences:

1. **Never compare floats with `==`.** Check `abs(a - b) < 1e-9` instead.
2. Real payment systems avoid the problem entirely by counting in integer *cents*.

</details>

### Booleans

Comparisons produce a `bool` — `True` or `False`. These are important, for example, for portfolio trading strategy signals ("is the price above the moving average?"):

In [ ]:
print(4 > 3)
print(4 <= 3)
print(5 == 4)      # equal?  (== compares; = assigns!)
print(5 != 4)      # not equal?
print((4 > 3) and (3 > 2))
print(True or False, not True)

### Strings

Text lives in `str` objects, defined with single or double quotes. Financial data are full of textual information, for example, ticker symbols, column names, file names, annual reports. Python is excellent at it:

In [ ]:
txt = 'financial modelling'
print(txt.capitalize())      # capitalize the first letter
print(txt.upper())           # ALL CAPS
print(txt.split())           # split into a list of words
print(txt.find('model'))     # position where 'model' starts
print(txt.find('2270'))      # predict: what does find() return when the text is NOT there?
print(txt.replace(' ', '@')) # replace all spaces

A different kind of strings are **f-strings**, for formatting numbers like a professional:

In [ ]:
portfolio_value = 1234567.8912
print(f"Portfolio value: {portfolio_value:,.2f} EUR")   # thousands separators, 2 decimals
print(f"Weight: {0.0834:.1%}")                          # as a percentage

---
## 2.2 Containers: tuple and list

| Structure | Defined with | Can change? | Used for |
|---|---|---|---|
| `tuple` | `( )` | no (immutable) | fixed records |
| `list` | `[ ]` | yes (mutable) | growing collections |
| `dict` | `{key: value}` | yes | lookups by name |
| `set` | `{ }` | yes | unique elements |

In [ ]:
myTuple = (2026, 2270, 'Financial Modelling')
print(type(myTuple))
print(myTuple[2])        # zero-based indexing: index 2 = THIRD element
print(type(myTuple[2]))

In [ ]:
myList = [2026, 2270, 'Financial Modelling']
print(myList[-1])        # negative index counts from the end

### Slicing

`a:b` slices indices `a, ..., b-1` — the end is **not included**. Before running the code, what will these commands output?

1. `myList[0:2]`
2. `myList[0:1]` — is this the same as `myList[0]`?
3. `myList[1:]`
4. `myList[3]`

In [ ]:
print(myList[0:2])
print(myList[0:1])   # a LIST containing one element...
print(myList[0])     # ...is not the same as the element itself
print(myList[1:])
# myList[3]          # uncomment and run: what happens, and what is the index of the last element?



### Multiplication trap

A colleague has daily returns saved in a list and wants to double them to account for leverage. Predict each output *before running*:

```python
returns = [1, 2, 3]
returns * 2        # what is this?
```

In [ ]:
returns = [1, 2, 3]
print(returns * 2)      # !!
print(returns + [4])    # and + also does not add numbers...

<details>
<summary> <b>Explanation</b></summary>

For lists, `*` means **repeat** and `+` means **concatenate** — no arithmetic at all. `[1,2,3] * 2` is `[1,2,3,1,2,3]`, not `[2,4,6]`. A backtest that doubles positions this way doesn't crash: it silently repeats the content twice. The fix is NumPy, where `*` finally means multiply.

</details>

### Why lists are not enough for finance

Lists allow mixed types (flexible, but slow and memory-hungry); slicing can't select "the 1st, 4th, and 5th element" or "all positive returns"; math doesn't work elementwise; and data is often 2-dimensional (assets × dates). Enter **NumPy**.

---
## 2.3 NumPy arrays

An `ndarray` is like a list, but all elements share one type, math works elementwise, indexing is far more powerful, and it can have many dimensions. We create arrays from lists:

In [ ]:
import numpy as np

myVec = np.array([1, 2, 3])              # 1D array ("vector")
print(myVec)
print(myVec * 2)                         # repeat or multiply?

In [ ]:
myMat = np.array([[1, 2, 3],
                  [5, 6, 7]])            # list of lists -> 2D array ("matrix")
print(myMat)

myOneRowMat = np.array([[1, 2, 3]])      # a 1 x 3 MATRIX -- note the double brackets
myOneColMat = myOneRowMat.transpose()    # switch rows and columns
print(myOneRowMat)
print(myOneColMat)

In [ ]:
# other ways to create arrays
print(np.zeros((2, 3)))    # 2 x 3 of zeros
print(np.ones((3, 4)))     # 3 x 4 of ones
print(np.eye(4))           # 4 x 4 identity matrix

### Practice: the missing maturity

Here are two functions that create evenly spaced arrays: `np.linspace(start, end, how_many)` and `np.arange(start, end, step)`. A student needs the four semi-annual coupon dates of a 2-year bond: 0.5, 1.0, 1.5, 2.0. Their code is below. **Is the code going to work? Run, explain the bug, fix it two ways.**

In [ ]:
maturities = np.arange(0.5, 2, 0.5)
print(maturities)          # where did the 2-year coupon go?

<details>
<summary> <b>Solution</b></summary>

`np.arange` **excludes the upper bound** — like slicing. The 2.0 never appears, and a bond just lost its final cash flow. Fixes:

```python
np.linspace(0.5, 2, 4)          # preferred: you say how many points you want
np.arange(0.5, 2 + 1e-10, 0.5)  # or push the bound just past 2
```

`.shape` tells you an array's size — checking `.shape` after creating an array is the quickest check there is:

```python
print(np.linspace(0.5, 2, 4).shape)    # (4,)
```

</details>

### Fancy indexing — selecting data like an analyst

Beyond single elements and slices, arrays accept a **list of positions** or a **boolean condition**. The boolean version is very helpful in this course:

In [ ]:
maturities = np.linspace(0.5, 2, 4)
rates = np.array([5.0, 5.8, 6.5, 6.8])

print(maturities[[1, 3]])              # positions 1 and 3
print(maturities[[0, -1]])             # first and last
print(maturities[maturities < 2])      # condition on the array itself
print(maturities[rates < 6])           # think first: condition on ONE array selecting from ANOTHER

That last line shows: *"give me the maturities at which the rate is below 6%"* — one readable line, no loop. Later we will have the same pattern *"give me the returns on days when the price was above the average"*, and it will help us to define a trading strategy.

### 2D indexing

`M[row, col]` — indices for each dimension, separated by a comma; `:` means "everything in this dimension":

In [ ]:
myMatrix = np.array([[1,  8, 7, 4],
                     [5,  6, 9, 0],
                     [3, 11, 0, 4]])
print(myMatrix[0, 1])       # row 0, column 1
print(myMatrix[2, -1])      # last row's last element
print(myMatrix[0:2, 2:4])   # predict the shape first: rows 0-1, columns 2-3
print(myMatrix[:, 2:3])     # all rows, column 2 -- as a column

### Practice: the portfolio that changed itself

A risk analyst extracts the first three maturities to work on, edits their copy — and then something odd happens. **Predict both printed lines before running.**

In [ ]:
maturities = np.array([1, 5, 9, 4, 5, 6])
first_maturities = maturities[0:3]

first_maturities[0] = 2          # edit "the copy"

print(first_maturities)
print(maturities)                # ...did the original survive?

<details>
<summary> <b>Solution</b></summary>

The original changed too! A NumPy slice is a **view**, not a copy — both names point at the same memory. Editing the "extract" silently corrupted the source data. In a real project this is a nightmare bug: nothing crashes, and the damage shows up three steps later in someone else's numbers.

The fix — ask for an independent copy explicitly:

```python
first_maturities = maturities[0:3].copy()
```

(Boolean indexing like `maturities[maturities < 5]` returns a copy, not a view — one more reason to use it.)

</details>

---
## 2.4 Mathematical operations on arrays

**Statistics** — every method can work on the whole array or along one dimension (`axis=0`: down the rows; `axis=1`: across the columns). If rows were dates and columns were assets, `axis=0` would give each asset's average over time:

In [ ]:
myMatrix = np.array([[1,  8, 7, 4],
                     [5,  6, 9, 0],
                     [3, 11, 0, 4]])
print(myMatrix.mean())                          # grand mean
print(myMatrix.mean(axis=0))                    # predict: how many numbers?
print(myMatrix.mean(axis=1))
print(myMatrix.mean(axis=1, keepdims=True))     # same, but kept as a column

**Arithmetic operations are applied elementwise** — `*` finally multiplies. Array–scalar operations broadcast the scalar to every element; array–array operations pair up matching elements:

In [ ]:
A = np.array([[1,  8, 7, 4],
              [5,  6, 9, 0],
              [3, 11, 0, 4]])
B = np.array([[9, 3,  5, 0],
              [4, 1, 10, 7],
              [5, 8,  3, 9]])
print(np.sqrt(A))     # elementwise square root
print(A * 2)          # compare with the list "surprise"
print(A + B)

### Broadcasting

Arrays of *different* shapes can sometimes combine: NumPy stretches ("broadcasts") the smaller one along its size-1 dimensions. Adding a 4-element vector to a 3×4 matrix adds it **to every row** — exactly what you want when, say, subtracting each asset's mean return from its column:

In [ ]:
v = np.array([2, 2, 2, 2])
print(A + v)          # v is added to every row

---
## 2.5 The speed race between lists and arrays

The claim was that lists are slow and NumPy is fast. Let's run an experiment. One million daily returns; we want to multiply them by 2.


In [ ]:
big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)

%timeit [x * 2 for x in big_list]     # pure-Python loop over a list
%timeit big_array * 2                 # NumPy, elementwise

Typically 50–200× faster — and the gap *widens* with more complex operations. **Advice: if you are writing a loop over data, first ask whether an array operation can do it.**

---
## 2.6 Exercise:  no loops allowed

Ten (fictional) closing prices of a stock:

```python
prices = np.array([100.0, 102.0, 101.5, 105.0, 103.0, 99.0, 98.5, 104.0, 107.0, 106.0])
```

Using only what you learned today — slicing, elementwise math, boolean indexing, statistics; **no `for` loops**:

1. Compute the 9 daily *simple returns*: $r_t = P_t/P_{t-1} - 1$. *(Hint: divide one slice of `prices` by another.)*
2. What was the worst daily return, and the average?
3. On how many days was the return positive? *(Hint: `True` counts as 1 -> try `.sum()` on a boolean array.)*
4. Select only the positive returns. What is their average, vs. the average of the negative ones?
5. In *one line*: the average return on days when the price was above 101 the previous day.



In [ ]:
# solution
prices = np.array([100.0, 102.0, 101.5, 105.0, 103.0, 99.0, 98.5, 104.0, 107.0, 106.0])



<details>
<summary> <b>Solution</b></summary>

```python
prices = np.array([100.0, 102.0, 101.5, 105.0, 103.0, 99.0, 98.5, 104.0, 107.0, 106.0])

# 1. slice arithmetic: elements 1..end divided by elements 0..end-1
returns = prices[1:] / prices[:-1] - 1

# 2.
print(f"worst: {returns.min():.2%}, average: {returns.mean():.2%}")

# 3. booleans sum as 0/1
print("positive days:", (returns > 0).sum())

# 4.
print(f"avg up-day:   {returns[returns > 0].mean():.2%}")
print(f"avg down-day: {returns[returns < 0].mean():.2%}")

# 5. condition on yesterday's price, applied to today's return
print(f"{returns[prices[:-1] > 101].mean():.2%}")
```

Item 5 is, structurally, a complete trading-signal evaluation — condition on yesterday's information, measure today's return.

</details>